# 3.5 — Introduction to LangGraph

The agent loop from notebooks 3.1–3.4 was a `while True` loop in Python.
**LangGraph** gives that loop a proper structure:

- **State** — a dict that flows through the graph
- **Nodes** — functions that transform the state
- **Edges** — connections between nodes (fixed or conditional)

```
START → agent_node → should_continue? → tool_node → agent_node → ... → END
                           ↓ (no tool calls)
                          END
```

Benefits over a raw while loop:
- Visualisable graph structure
- Built-in streaming
- Human-in-the-loop checkpoints
- Persistent state between runs

In [ ]:
!pip install langchain langchain-ollama langgraph --quiet

## Part 1 — Simplest LangGraph: Two Nodes

In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

# 1. Define the state — what flows through the graph
class MyState(TypedDict):
    text: str
    result: str

# 2. Define nodes — functions that transform state
def uppercase_node(state: MyState) -> MyState:
    """Converts text to uppercase."""
    return {'text': state['text'], 'result': state['text'].upper()}

def exclaim_node(state: MyState) -> MyState:
    """Adds exclamation marks."""
    return {'text': state['text'], 'result': state['result'] + '!!!'}

# 3. Build the graph
graph = StateGraph(MyState)
graph.add_node('uppercase', uppercase_node)
graph.add_node('exclaim',   exclaim_node)

graph.add_edge(START,       'uppercase')
graph.add_edge('uppercase', 'exclaim')
graph.add_edge('exclaim',   END)

app = graph.compile()

# 4. Run
result = app.invoke({'text': 'hello world', 'result': ''})
print('Input :', 'hello world')
print('Output:', result['result'])

## Part 2 — Conditional Edges

Conditional edges let the graph branch based on state — just like `if/else`.

In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Literal

class ReviewState(TypedDict):
    text: str
    sentiment: str
    reply: str

from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage

llm = ChatOllama(model='llama3.1', temperature=0)

def classify_node(state: ReviewState) -> ReviewState:
    """Classify review as positive or negative."""
    prompt = f'Is this review positive or negative? Reply with one word only (positive/negative).\n\nReview: {state["text"]}'
    result = llm.invoke([HumanMessage(content=prompt)]).content.lower().strip()
    sentiment = 'positive' if 'positive' in result else 'negative'
    return {**state, 'sentiment': sentiment}

def positive_reply_node(state: ReviewState) -> ReviewState:
    reply = 'Thank you for the great feedback! We are thrilled you enjoyed it.'
    return {**state, 'reply': reply}

def negative_reply_node(state: ReviewState) -> ReviewState:
    reply = 'We are sorry to hear about your experience. Our team will reach out to resolve this.'
    return {**state, 'reply': reply}

def route_by_sentiment(state: ReviewState) -> Literal['positive_reply', 'negative_reply']:
    """Routing function — decides which node to go to next."""
    return 'positive_reply' if state['sentiment'] == 'positive' else 'negative_reply'

# Build graph with conditional routing
graph = StateGraph(ReviewState)
graph.add_node('classify',       classify_node)
graph.add_node('positive_reply', positive_reply_node)
graph.add_node('negative_reply', negative_reply_node)

graph.add_edge(START, 'classify')
graph.add_conditional_edges('classify', route_by_sentiment)
graph.add_edge('positive_reply', END)
graph.add_edge('negative_reply', END)

review_app = graph.compile()

# Test
reviews = [
    'The food was absolutely amazing, best pizza I have ever had!',
    'Terrible service, waited 2 hours and the order was wrong.',
]

for review in reviews:
    result = review_app.invoke({'text': review, 'sentiment': '', 'reply': ''})
    print(f'Review    : {review[:60]}...')
    print(f'Sentiment : {result["sentiment"]}')
    print(f'Reply     : {result["reply"]}')
    print()

## Part 3 — Agent Loop as a LangGraph

Now let's rewrite the agent from 3.2 as a proper LangGraph.

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage
from langchain_core.tools import tool
from typing import TypedDict, Annotated, Sequence, Literal
import operator

# Tools
@tool
def add(a: int, b: int) -> int:
    """Adds two numbers."""
    return a + b

@tool
def multiply(a: int, b: int) -> int:
    """Multiplies two numbers."""
    return a * b

@tool
def get_capital(country: str) -> str:
    """Returns the capital city of a country."""
    caps = {'france': 'Paris', 'japan': 'Tokyo', 'india': 'New Delhi', 'germany': 'Berlin'}
    return caps.get(country.lower(), f'Capital of {country} not found')

tools = [add, multiply, get_capital]

llm_tools = llm.bind_tools(tools)

# State — uses a list of messages with append reducer
class AgentState(TypedDict):
    messages: Annotated[list[BaseMessage], operator.add]

SYSTEM = SystemMessage(content='You are a helpful assistant. Use tools when needed.')

# Nodes
def agent_node(state: AgentState) -> AgentState:
    """Call the LLM."""
    messages = [SYSTEM] + state['messages']
    response = llm_tools.invoke(messages)
    return {'messages': [response]}

def should_continue(state: AgentState) -> Literal['tools', '__end__']:
    """If the last message has tool calls, go to tools node. Otherwise end."""
    last = state['messages'][-1]
    if hasattr(last, 'tool_calls') and last.tool_calls:
        return 'tools'
    return '__end__'

# Build graph
tool_node = ToolNode(tools)

agent_graph = StateGraph(AgentState)
agent_graph.add_node('agent', agent_node)
agent_graph.add_node('tools', tool_node)

agent_graph.add_edge(START, 'agent')
agent_graph.add_conditional_edges('agent', should_continue)
agent_graph.add_edge('tools', 'agent')  # after tools, go back to agent

agent_app = agent_graph.compile()

print('Agent graph compiled.')

# Run
def ask(question: str):
    result = agent_app.invoke({'messages': [HumanMessage(content=question)]})
    print(f'Q: {question}')
    print(f'A: {result["messages"][-1].content}')
    print()

ask('What is 25 + 17?')
ask('What is the capital of India?')
ask('What is 12 multiplied by 8, then add 5?')

## Part 4 — Streaming Agent Steps

In [ ]:
# Stream lets you see each node's output as it happens
print('Streaming agent execution for: "What is (15 + 5) multiplied by 3?"')
print()

for step in agent_app.stream({'messages': [HumanMessage(content='What is (15 + 5) multiplied by 3?')]}):
    node_name = list(step.keys())[0]
    messages  = step[node_name]['messages']
    last_msg  = messages[-1]
    print(f'[{node_name}]')
    if hasattr(last_msg, 'tool_calls') and last_msg.tool_calls:
        for tc in last_msg.tool_calls:
            print(f'  → calls {tc["name"]}({tc["args"]})')
    else:
        print(f'  → {last_msg.content}')
    print()

## Summary

| Concept | Description |
|---------|-------------|
| `StateGraph` | The graph builder |
| `State` | TypedDict that flows through nodes |
| `add_node` | Register a function as a node |
| `add_edge` | Fixed connection between nodes |
| `add_conditional_edges` | Branch based on state |
| `ToolNode` | Pre-built node that executes tool calls |
| `.compile()` | Compiles graph into runnable app |
| `.stream()` | Run graph and yield each node's output |